## Odia Sentence Splitting & Inference Preparation
This notebook executes the second stage of the data pipeline, transforming the cleaned paragraph-level text into a sentence-level format optimized for **sequence-to-sequence (Seq2Seq)** model inference.

### Key Achievements:
* **Linguistic Segmentation:**
  * Utilizes the `indic-nlp-library` to accurately split Odia paragraphs into individual sentences.
  * This is superior to standard punctuation splitting (like .split('.')) because it handles Odia-specific punctuation (e.g., the danda `|` used as a full stop) correctly.
* **Data Explosion:**
  * Successfully transformed **100,000 raw documents** (from the previous cleaning step) into **1,455,310 individual sentences**.
  * This massive increase in data points provides a rich corpus for evaluating or training translation models.
* **Model-Ready Formatting:**
  * **Prefix Injection:** Automatically prepends the task prompt `"translate Odia to German: "` to every sentence.
  * **Filtering:** Implements length constraints (min 15 characters, max 200 words) to discard noise, ensuring only meaningful sentences are passed to the model.
  * **Final Output:** Generates a structured JSONL file (`varta_02_sentences_inference.jsonl`) ready for direct loading into Hugging Face `Datasets`.

### Workflow Context:
* **Input:** `varta_02_cleaned.jsonl` (Cleaned paragraph text)
* **Process:** Tokenization $\to$ Splitting $\to$ Prefixing
* **Output:** `varta_02_sentences_inference.jsonl` (Ready for NLLB Inference)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Install the necessary library for Indian language text processing
!pip install -q indic-nlp-library

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 10.0 MB/s eta 0:00:00


In [ ]:
import json
import os
from tqdm import tqdm
from indicnlp.tokenize import sentence_tokenize

In [ ]:
# --- CONFIGURATION ---
INPUT_FILE = "/content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/varta_02_cleaned.jsonl"
OUTPUT_FILE = "/content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/varta_02_sentences_inference.jsonl"
PREFIX_ORI_TO_DEU = "translate Odia to German: "

In [ ]:
def split_and_prepare_data():
    """
    Processes raw paragraph-level data into a cleaner, sentence-level dataset.

    This function iterates through the input JSONL file, decomposing paragraphs
    into individual sentences using the IndicNLP library (specifically for Odia).
    It applies length-based filtering and formats the data for model training
    by adding task-specific prefixes.

    Operations:
        1. Reads JSONL records from the global `INPUT_FILE`.
        2. Tokenizes paragraphs into sentences handling Odia punctuation.
        3. Filters sentences to retain those with 5 to 100 words.
        4. Appends the `PREFIX_ORI_TO_DEU` for translation task prompting.
        5. Writes the processed records to `OUTPUT_FILE`.

    Global Dependencies:
        INPUT_FILE (str): Path to the raw JSONL input.
        OUTPUT_FILE (str): Path to the destination file.
        PREFIX_ORI_TO_DEU (str): The prompt prefix (e.g., 'translate Odia to German: ').

    Returns:
        None: Results are written directly to disk.
    """
    print(f"Reading from: {INPUT_FILE}")
    processed_sentences = []

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        # Read all lines
        lines = f.readlines()

    print("Splitting paragraphs into sentences...")
    sentence_id = 1

    for line in tqdm(lines):
        if not line.strip(): continue

        record = json.loads(line)
        paragraph = record.get("text", "")
        source = record.get("source", "")

        if not paragraph: continue

        # Split Paragraph into Sentences using IndicNLP
        # This handles Odia-specific punctuation ('।', question marks, etc.)
        sentences = sentence_tokenize.sentence_split(paragraph, lang='or')

        for sent in sentences:
            sent = sent.strip()

            # Filter: Keep sentences between 5 and 100 words (approx 200 tokens)
            # This ensures NLLB handles them perfectly.
            word_count = len(sent.split())
            if 5 <= word_count <= 100:

                new_record = {
                    "id": sentence_id,
                    "input_text": PREFIX_ORI_TO_DEU + sent, # Add the training prefix
                    "raw_odia": sent,
                    "source": source,
                    "original_paragraph_snippet": paragraph[:50] + "..." # Just for reference
                }
                processed_sentences.append(new_record)
                sentence_id += 1

    # Save the sentence-level dataset
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f_out:
        for item in processed_sentences:
            f_out.write(json.dumps(item, ensure_ascii=False) + '\n')

    print(f"\n✅ Created {len(processed_sentences)} individual sentence records.")
    print(f"Saved to: {OUTPUT_FILE}")

split_and_prepare_data()

Reading from: /content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/varta_02_cleaned.jsonl
Splitting paragraphs into sentences...


100%|██████████| 100000/100000 [00:11<00:00, 8914.22it/s]



✅ Created 1455310 individual sentence records.
Saved to: /content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/varta_02_sentences_inference.jsonl


In [ ]:
print(f"--- First 5 Records from {OUTPUT_FILE.split('/')[-1]} ---")

with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    for i in range(5):
        line = f.readline()
        if not line: break

        # Parse JSON and print clearly
        record = json.loads(line)
        print(json.dumps(record, indent=4, ensure_ascii=False))
        print("-" * 50)

--- First 5 Records from varta_02_sentences_inference.jsonl ---
{
    "id": 1,
    "input_text": "translate Odia to German: ଲକ୍ଷ୍ନୋ,୧୪।୨: ପ୍ରଥମ ପର୍ଯ୍ୟାୟ ପରେ ସୋମବାର ଉତ୍ତରପ୍ରଦେଶର ୫୫ ଆସନରେ ଦ୍ୱିତୀୟ ପର୍ଯ୍ୟାୟ ଭୋଟ ଗ୍ରହଣ ଚାଳିଛି।",
    "raw_odia": "ଲକ୍ଷ୍ନୋ,୧୪।୨: ପ୍ରଥମ ପର୍ଯ୍ୟାୟ ପରେ ସୋମବାର ଉତ୍ତରପ୍ରଦେଶର ୫୫ ଆସନରେ ଦ୍ୱିତୀୟ ପର୍ଯ୍ୟାୟ ଭୋଟ ଗ୍ରହଣ ଚାଳିଛି।",
    "source": "varta",
    "original_paragraph_snippet": "ଲକ୍ଷ୍ନୋ,୧୪।୨: ପ୍ରଥମ ପର୍ଯ୍ୟାୟ ପରେ ସୋମବାର ଉତ୍ତରପ୍ରଦେ..."
}
--------------------------------------------------
{
    "id": 2,
    "input_text": "translate Odia to German: ମତଦାନ ସକାଳ ୭ଟାରୁ ଆରମ୍ଭ ହୋଇଥିବା ବେଳେ ଅପରାହ୍ନ ସାଢେ ୩ଟା ସୁଦ୍ଧା ୫୧.୯୩% ମତଦାନ ହୋଇଛି।",
    "raw_odia": "ମତଦାନ ସକାଳ ୭ଟାରୁ ଆରମ୍ଭ ହୋଇଥିବା ବେଳେ ଅପରାହ୍ନ ସାଢେ ୩ଟା ସୁଦ୍ଧା ୫୧.୯୩% ମତଦାନ ହୋଇଛି।",
    "source": "varta",
    "original_paragraph_snippet": "ଲକ୍ଷ୍ନୋ,୧୪।୨: ପ୍ରଥମ ପର୍ଯ୍ୟାୟ ପରେ ସୋମବାର ଉତ୍ତରପ୍ରଦେ..."
}
--------------------------------------------------
{
    "id": 3,
    "input_text": "translate Odia to German: ଦ୍ବିତୀୟ ପର୍ଯ୍ୟାୟରେ ୫୮୪ ପ୍ରାର